In [ ]:
import pandas as pd

In [ ]:
# Read the data 
df = pd.read_excel("data/Marklist.xlsx", header=None)

# Extract the required data from the excel sheet.
df.columns = df.iloc[7] # header
df = df.iloc[8:72] # content
df.columns = df.columns.astype(str).str.strip()
df = df.reset_index(drop=True)
df.columns.name = None

In [ ]:
df.head() # reference

In [ ]:
df.columns # reference

In [ ]:
# checking for inconsistencies and NaN handling.
df["SGPA"] = pd.to_numeric(df["SGPA"],errors="coerce")
df["SGPA"].isna().sum()

In [ ]:
df = df.copy()

# Create response flag safely
df.loc[:, "Responded"] = df["SGPA"].notna()

# Split without overwriting df
responded_df = df[df["Responded"] == True]
non_responded_df = df[df["Responded"] == False]

# Total 64 students
print("Responded:", len(responded_df)) # 44 filled the sheet
print("Not Responded:", len(non_responded_df)) # 20 not filled
responded_df["SGPA"].isna().sum() # 0 - No inconsistencies => After handling the inconsistencies

In [ ]:
# Handling Inconsistencies in subject scores

# Only extracting the subject columns.
non_subject_cols = [
    "S.No", "Register Number", "NAME",
    "SGPA", "No.of Arr", "Pass/Fail", "Rank", "Responded"
]

subject_cols = [col for col in df.columns if col not in non_subject_cols]
df = df.copy()

df["No.of Arr"] = df[subject_cols].apply(
    lambda row: (row.astype(str).str.upper() == 'U').sum(),
    axis = 1
)

# Filling all the Non-filled columns.
df.loc[df["No.of Arr"] > 0, "SGPA"] = pd.NA
df.loc[df["No.of Arr"] == 0, "Pass/Fail"] = "Pass"
df.loc[df["No.of Arr"] > 0, "Pass/Fail"] = "Fail"

# Ranking the students based on SGPA
df["Rank"] = pd.NA

df.loc[df["No.of Arr"] == 0, "Rank"] = (
    df.loc[df["No.of Arr"] == 0, "SGPA"]
      .rank(ascending=False, method="dense")
      .astype("Int64")   
)


In [ ]:
df # reference

In [ ]:
# segreggating students eligible for ranking
rank_base = responded_df[responded_df["No.of Arr"] == 0].copy()
rank_base["Rank"] = (
    rank_base["SGPA"]
    .rank(ascending=False, method="dense")
    .astype("Int64")
)

df["Rank"] = pd.NA

df.loc[rank_base.index,"Rank"] = rank_base["Rank"]

df_sorted = df.sort_values(
    by = "Rank",
    ascending = True,
    na_position = "last",
).reset_index(drop = True)
df_sorted[["NAME", "SGPA", "No.of Arr", "Responded", "Rank"]].head(10)

In [ ]:
with open("output/ranked_list.csv","w",encoding="utf-8") as f:
    f.write(",".join(df_sorted.columns) + "\n")
    for i, row in df_sorted.iterrows():
        f.write(",".join(map(str,row.values)) + "\n")

In [ ]:
# top 10 rank holders
top_10 = df_sorted[df_sorted["Rank"].notna()].head(10)

top_10[[
    "Rank",
    "Register Number",
    "NAME",
    "SGPA",
    "No.of Arr"
]]

In [ ]:
# Rank vs SGPA analysis

import matplotlib.pyplot as plt
rank_only = df_sorted[df_sorted["Rank"].notna()]
plt.figure(figsize=(6,4))
plt.scatter(rank_only["Rank"],rank_only["SGPA"],s=70)
plt.xlabel("Rank")
plt.ylabel("SGPA")
plt.title("SGPA vs Rank")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# calculating the pass fail frequencies using bar chat.
df["Pass/Fail"].value_counts().plot(kind="bar")

plt.title("Pass vs Fail Count")
plt.xlabel("Result")
plt.ylabel("Number of Students")
plt.tight_layout()
plt.show()

In [ ]:
# Arrear per subject
arrear_by_subject = (
    df[subject_cols]
    .apply(lambda col: (col.astype(str).str.upper() == "U").sum())
    .sort_values(ascending=False)
)

arrear_by_subject # reference

In [ ]:
# ploting the number of arrears 
arrear_by_subject.plot(kind="bar", figsize=(8,4))

plt.title("Subject-wise Arrear Count")
plt.xlabel("Subject Code")
plt.ylabel("Number of Arrears")
plt.tight_layout()
plt.show()

In [ ]:
arrear_by_subject = (
    df[subject_cols]
    .apply(lambda col: (col.astype(str).str.upper() == "U").sum())
)

In [ ]:
with open("output/subject_analysis/subject_wise_failures.csv", "w", encoding="utf-8") as f:
    f.write("Subject,Failure_Count\n")
    for subject, count in arrear_by_subject.items():
        f.write(f"{subject},{count}\n")

In [ ]:
plt.figure(figsize=(10,5))
plt.bar(arrear_by_subject.index, arrear_by_subject.values)
plt.xlabel("Subject")
plt.ylabel("Number of Failures")
plt.title("Subject-wise Failure Count")
plt.xticks(rotation=45)
plt.tight_layout()

plt.savefig("output/subject_analysis/subject_wise_failures.png")
plt.close()